# Build a multi-agent system: the claim desk

*Applied AI for Business · University of Amsterdam*

In this notebook you run a small team of AI agents. Together they handle a hail-damage claim for a fictional insurer. **You do not need to know how to program.** Every step is explained.

**The team**

```
TRIGGER   a claim email arrives
AGENT 1   intake: reads the email and lists the facts
CHECK     plain code: are the facts complete? If not, the chain stops
AGENT 2   policy: looks up the policy with a TOOL and decides
AGENT 3   letter: writes the reply to the customer
PERSON    you: approve the letter or not. Nothing is sent automatically
```

**Follow the work from agent to agent.** Every agent prints its answer in a box. That answer, word for word, is what the next agent receives. So you can see exactly how the work is passed on.

**Your own email.** In Step 6 you can also upload a claim email of your own (.txt, .eml, .msg, .pdf, .docx or .html) or paste one. Use invented emails only.

**How to use this notebook**

- A notebook has two kinds of blocks: **text blocks** like this one, and **code blocks** (the grey ones).
- To run a code block: click on it, then click the ▶ button on its left, or press **Shift + Enter**.
- Run the blocks **in order, from top to bottom**. If something goes wrong, choose *Runtime → Restart session* and start again at the top.
- **Some blocks wait for you:** Step 1 may ask whether it may use your secret (click *Grant access*), the trigger waits until you choose a file, and the person block waits for *yes* or *no*. While one block waits, no other block can run: the next block you start shows a spinning circle. Answer the block that waits, or stop it with its ■ button (or *Runtime → Interrupt execution*).
- In a code block, text after a `#` is a **comment**: a note for people. The computer ignores it.
- Some blocks show a form on the right (a tick box or a menu). You can use the form instead of changing the code.

**Costs**

- **Demo mode** (the default): free, no account needed. The agents give fixed example answers.
- **Real mode**: uses a real AI model. You need your own OpenAI API key. One run costs well under 1 cent.

All people, policies and claims in this notebook are invented.

## Step 0 · Choose demo mode or real mode

**What this block does:** it stores one choice: demo mode or real mode.

**What you will see:** one line that confirms your choice.

**Cost:** free.

**Want real mode?**
1. Get an API key from your own OpenAI account. Set a monthly spending limit in that account first.
2. In Colab, click the **key icon** in the left bar. Choose *Add new secret*. Name: `OPENAI_API_KEY`. Value: your key. Switch on *Notebook access*.
3. Untick `DEMO_MODE` in the form on the right, and run the block.

⚠️ Never paste your key into a code block. Anyone who sees the notebook would see your key. A Colab secret stays hidden.

In [ ]:
# @title Step 0 · Demo mode or real mode?
DEMO_MODE = True  # @param {type:"boolean"}

# A "variable" is a name that holds a value.
# DEMO_MODE holds True (demo mode) or False (real mode).
if DEMO_MODE:
    print("Demo mode: the agents give fixed example answers. Free.")
else:
    print("Real mode: the agents use a real OpenAI model. You need the secret OPENAI_API_KEY.")

## Step 1 · Connect to the model, and set the budget

**What this block does:**
- It sets the **budget** for one run: how many calls to the model are allowed, and how much a run may cost.
- In real mode, it installs the OpenAI package (software that lets Python send messages to OpenAI's models) and connects with your key from the secret.

**What you will see:** in demo mode, *No connection needed*. In real mode, *Connected to OpenAI*.

**Real mode, the first time:** Colab may ask whether this notebook may use your secret `OPENAI_API_KEY`. Click *Grant access*. Until you do, this block waits, and no other block can run.

**Cost:** free.

In [ ]:
# @title Step 1 · Connect and set the budget

# --- Which model, and what it costs (OpenAI's price list, November 2026) ---
MODEL = "gpt-6-luna"             # a small, cheap model
PRICE_INPUT_PER_MILLION = 0.10   # dollars per 1 million tokens the model READS (a token is about 3/4 of a word)
PRICE_OUTPUT_PER_MILLION = 0.50  # dollars per 1 million tokens the model WRITES (including its thinking)

# --- The budget for one run: the program stops if it would go over ---
MAX_CALLS = 6                    # a normal run needs 4 calls to the model
MAX_COST_USD = 0.05              # a normal run costs well under 1 cent

if DEMO_MODE:
    client = None                # in demo mode we do not connect to anything
    print("No connection needed in demo mode.")
else:
    !pip install -q "openai>=3.51,<4"
    # The line above installs the OpenAI package. The "!" means: run this as a command, not as Python.
    from google.colab import userdata         # Colab's safe place for secrets
    from openai import OpenAI                 # the tool that talks to OpenAI
    try:
        key = userdata.get("OPENAI_API_KEY")  # read your key from the secret, never from the code
    except Exception:
        raise SystemExit("No secret OPENAI_API_KEY found, or 'Notebook access' is off. See Step 0.")
    client = OpenAI(api_key=key)              # the connection, used by every agent below
    print("Connected to OpenAI. Model:", MODEL)

## Step 2 · The data: claim emails and a policy database

**What this block does:** it stores three invented claim emails and a small invented database with two insurance policies.
- Claim 1: hail damage to a roof, with a home insurance policy.
- Claim 2: hail damage to a roof, but the customer only has contents insurance.
- Claim 3: a vague email without a policy number.

**What you will see:** a short list of what was stored.

**Cost:** free.

**Try later:** change the text of an email, run this block again, and run the team again (real mode).

**Your own email:** in Step 6 you can also upload or paste a claim email of your own instead.

In [ ]:
# @title Step 2 · The claim emails and the policy database

# The three claim emails. Each email has a name (for example "claim_1_covered") and a text.
CLAIMS = {
    "claim_1_covered": """
From: Sanne de Vries
Subject: Hail damage to my roof

Dear claims desk,

During the hailstorm on 3 November 2026, hailstones broke six roof tiles and cracked the skylight above our stairs.
Rain is now coming in. A roofer came by yesterday. His quote for new tiles and a new skylight is EUR 1,850.

My policy number is DH-1001.

Kind regards,
Sanne de Vries
""",
    "claim_2_not_covered": """
From: Mehmet Yilmaz
Subject: Roof and skylight damaged by hail

Hello,

On 3 November 2026 a heavy hailstorm hit our street. The hail damaged our roof and broke the skylight in the attic.
The repair will cost about EUR 2,400 according to the builder.

Policy number: DH-2002.

Thank you,
Mehmet Yilmaz
""",
    "claim_3_missing_info": """
From: Lisa
Subject: storm

hi, after the storm last week there is water coming through the ceiling.
can you help me? what do i need to do?

Lisa
""",
}

# The policy database: two invented policies. Each policy has a number (for example "DH-1001").
POLICIES = {
    "DH-1001": {
        "holder": "Sanne de Vries",
        "product": "Home insurance: building and contents",
        "covers": [
            "storm",
            "hail",
            "fire",
            "water damage from a burst pipe"
        ],
        "does_not_cover": [
            "wear and tear",
            "damage caused by poor maintenance"
        ],
        "deductible_eur": 250,
        "max_payout_eur": 25000,
        "valid_until": "2027-12-31"
    },
    "DH-2002": {
        "holder": "Mehmet Yilmaz",
        "product": "Contents insurance only (the building itself is not insured)",
        "covers": [
            "theft",
            "fire",
            "storm or hail damage to belongings inside the home"
        ],
        "does_not_cover": [
            "damage to the building itself, such as the roof, windows or skylights"
        ],
        "deductible_eur": 100,
        "max_payout_eur": 10000,
        "valid_until": "2027-06-30"
    }
}

print("Stored", len(CLAIMS), "claim emails:", ", ".join(CLAIMS))
print("Stored", len(POLICIES), "policies:", ", ".join(POLICIES))

## Step 3 · The agents' instructions

An agent is mostly its **instructions**: a text that tells the model who it is and what it must do. Below are the instructions of the three agents. The model is the same for all three. Only the instructions differ.

**What these blocks do:** each one stores the instructions of one agent under a name, for example `INTAKE_INSTRUCTIONS`.

**What you will see:** one short confirmation per block.

**Cost:** free.

**Try later:** change the words, run the block again, then run the team again (real mode only: in demo mode the answers are fixed).

In [ ]:
# @title Agent 1 · intake: its instructions

# Everything between the triple quotes is the text the model receives as its instructions.
# The agent answers in plain lines that people can read. The check and agent 2 read the same lines.
INTAKE_INSTRUCTIONS = """
You are the INTAKE AGENT of the claims desk of a home insurer.

You read one claim email from a customer and list the facts for your colleagues.

Answer with exactly these six lines, and nothing else:
Customer name: ...
Policy number: ...
Date of damage: ...
What happened: ...
Damage: ...
Estimated cost: ...

Write the estimated cost in euros, for example: EUR 1,850.
Write "unknown" when the email does not say something. Do not guess.
"""

print("Stored the instructions of the intake agent.")

In [ ]:
# @title Agent 2 · policy: its instructions

# Everything between the triple quotes is the text the model receives as its instructions.
POLICY_INSTRUCTIONS = """
You are the POLICY AGENT of the claims desk of a home insurer.

You get the facts of a claim, as listed by the intake agent.

1. Use the tool lookup_policy to find the customer's policy.
2. Decide: COVERED, NOT COVERED or NEED MORE INFORMATION.
   Base your decision only on the policy that the tool returns.
3. If covered, the payout is the estimated cost minus the deductible, and never more than the maximum payout.

Answer with exactly these three lines, and nothing else:
Decision: ...
Reason: ... (one or two short sentences, in plain English)
Payout: ... (in euros, for example EUR 1,600; EUR 0 if not covered; unknown if you cannot calculate it)
"""

print("Stored the instructions of the policy agent.")

In [ ]:
# @title Agent 3 · letter: its instructions

# Everything between the triple quotes is the text the model receives as its instructions.
LETTER_INSTRUCTIONS = """
You are the LETTER AGENT of the claims desk of a home insurer.

You get the facts of a claim (from the intake agent) and the decision (from the policy agent).

Write a short, friendly reply to the customer, in English:
- Start with the line: DRAFT - to be checked by a claims handler
- Address the customer by name.
- Explain the decision and the reason in plain words.
- Mention the payout only if the decision is COVERED.
- Promise nothing that is not in the decision.
- At most 120 words.
- Sign as: Claims desk, Demo Home Insurance (a fictional insurer)
"""

print("Stored the instructions of the letter agent.")

## Step 4 · The tool

A **tool** is a normal piece of code that does one job. Here it looks up a policy in the database.

The model **cannot run code itself**. It can only *ask* our program to run the tool. Our program runs it and sends the answer back to the model. That is how an agent acts.

**What this block does:**
- It writes down the tool (a *function*: a recipe with a name, here `lookup_policy`).
- It writes a short description of the tool for the model, so the model knows the tool exists and what to send to it.
- It tests the tool once.

**What you will see:** the policy DH-1001, found by the tool.

**Cost:** free.

In [ ]:
# @title Step 4 · The tool: look up a policy
import json   # a standard tool for reading and writing data in JSON format

# The tool itself: a recipe (a "function"). Lines starting with "def" write down a recipe.
def lookup_policy(policy_number):
    """Look up a policy in the database. If the number is unknown, say so."""
    return POLICIES.get(policy_number, {"error": "No policy found with number " + str(policy_number)})

# The description the model receives: the tool's name, what it does, and what to send to it.
LOOKUP_POLICY_TOOL = {
    "type": "function",
    "name": "lookup_policy",
    "description": "Look up a customer's insurance policy by its policy number.",
    "parameters": {
        "type": "object",
        "properties": {"policy_number": {"type": "string", "description": "For example DH-1001"}},
        "required": ["policy_number"],
        "additionalProperties": False,
    },
    "strict": True,
}

# A test: use the recipe once and show the result.
print("Test of the tool, policy DH-1001:")
print(json.dumps(lookup_policy("DH-1001"), indent=2))

## Step 5 · The engine room

This block holds the machinery that every agent uses:
- **the budget guard**: it counts the calls and the cost, and stops a run that would go over the budget;
- **`ask_model`**: sends one message to the model and returns the answer;
- **`show_box`**: prints an agent's output in a box, with where it goes next;
- **`read_lines`**: lets plain code read lines such as *Policy number: DH-1001*;
- **`read_email_file`**: turns an uploaded email (.txt, .eml, .msg, .pdf, .docx or .html) into plain text;
- **`letter_to_pdf`**: saves the approved letter as a PDF file;
- **the fixed answers for demo mode.**

You do not need to read this block. The code is hidden; click *Show code* if you are curious.

**What you will see:** *Engine room ready.*

**Cost:** free.

In [ ]:
# @title Step 5 · The engine room (click "Show code" to see the code)
import io, re, textwrap, html as html_tools

calls = []   # the budget guard's notebook: one line per call to the model
MAX_EMAIL_CHARS = 8000   # very long emails are cut, to protect the budget

def reset_budget():
    """Start a new run: forget the calls of the previous run."""
    calls.clear()

def total_cost():
    return sum(cost for agent, read, written, cost in calls)

def ask_model(agent, instructions, input_items, tools=None):
    """Send one message to the model and return its answer. The budget guard checks before and after."""
    if len(calls) >= MAX_CALLS:
        raise RuntimeError(f"Budget guard: already {len(calls)} calls in this run (limit {MAX_CALLS}). "
                           "Start a new run with the trigger block.")
    answer = client.responses.create(
        model=MODEL,
        instructions=instructions,          # the agent's instructions (Step 3)
        input=input_items,                  # the work: an email, facts, or a decision
        tools=tools or [],                  # the tools the agent may ask for (only agent 2 has one)
        max_output_tokens=2000,             # the most one answer may use
        reasoning={"effort": "low"},        # less thinking: fewer tokens, so cheaper
    )
    read, written = answer.usage.input_tokens, answer.usage.output_tokens
    cost = (read * PRICE_INPUT_PER_MILLION + written * PRICE_OUTPUT_PER_MILLION) / 1_000_000
    calls.append((agent, read, written, cost))
    if total_cost() > MAX_COST_USD:
        raise RuntimeError(f"Budget guard: this run cost ${total_cost():.4f}, over the limit of ${MAX_COST_USD}.")
    return answer

def show_box(title, text, goes_to=""):
    """Print a text in a box, with a title and where the text goes next."""
    width = 80
    print("┌" + "─" * width)
    print("│ " + title)
    if goes_to:
        print("│ → " + goes_to)
    print("├" + "─" * width)
    for line in str(text).strip().split("\n"):
        for part in (textwrap.wrap(line, width - 2) or [""]):
            print("│ " + part)
    print("└" + "─" * width)

def read_lines(text):
    """Read lines like 'Policy number: DH-1001' into labels and values, so plain code can check them."""
    found = {}
    for line in str(text).split("\n"):
        line = line.replace("**", "").strip().lstrip("-• ").strip()
        if ":" in line:
            label, value = line.split(":", 1)
            found[label.strip().lower()] = value.strip()
    return found

def is_unknown(value):
    """Empty, or 'unknown' and the like, counts as missing."""
    return value.strip().strip(".").lower() in ("", "unknown", "not stated", "not given", "none", "n/a", "null")

# ---- Reading your own email: turn an uploaded file into plain text ----
def need(package, module):
    """Load an extra package that a file type needs. Install it first if Colab does not have it yet."""
    import importlib, subprocess, sys
    try:
        return importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        return importlib.import_module(module)

def html_to_text(page):
    """Remove the layout code from an HTML email and keep the text."""
    page = re.sub(r"(?is)<(head|script|style)\b.*?</\1>", "", page)
    page = re.sub(r"(?i)<br\s*/?>|</p>|</div>|</tr>|</li>|</h\d>", "\n", page)
    text = html_tools.unescape(re.sub(r"<[^>]+>", "", page))
    return re.sub(r"\n\s*\n+", "\n\n", text).strip()

def with_header(sender, subject, body):
    """Put the sender and the subject (if known) above the text of the email."""
    header = [label + ": " + str(value) for label, value in (("From", sender), ("Subject", subject)) if value]
    return "\n".join(header + [""] + [body.strip()]).strip()

def read_email_file(filename, data):
    """Turn an uploaded email into plain text. Works for .txt, .md, .eml, .msg, .pdf, .docx and .html."""
    kind = filename.lower().rsplit(".", 1)[-1] if "." in filename else ""
    if kind in ("txt", "md", "text"):
        for encoding in ("utf-8", "cp1252", "latin-1"):
            try:
                return data.decode(encoding)
            except UnicodeDecodeError:
                pass
    if kind in ("html", "htm"):
        return html_to_text(data.decode("utf-8", errors="replace"))
    if kind == "eml":                                   # an email saved from Gmail, Apple Mail, Thunderbird ...
        from email import policy
        from email.parser import BytesParser
        message = BytesParser(policy=policy.default).parsebytes(data)
        part = message.get_body(preferencelist=("plain", "html"))
        body = "" if part is None else part.get_content()
        if part is not None and part.get_content_type() == "text/html":
            body = html_to_text(body)
        return with_header(message["from"], message["subject"], body)
    if kind == "msg":                                   # an email saved from Outlook
        extract_msg = need("extract-msg", "extract_msg")
        import tempfile, os
        with tempfile.NamedTemporaryFile(suffix=".msg", delete=False) as f:
            f.write(data)
        message = extract_msg.openMsg(f.name)
        body = message.body or html_to_text((message.htmlBody or b"").decode("utf-8", errors="replace"))
        text = with_header(message.sender, message.subject, body)
        message.close()
        os.remove(f.name)
        return text
    if kind == "pdf":
        pypdf = need("pypdf", "pypdf")
        return "\n".join(page.extract_text() or "" for page in pypdf.PdfReader(io.BytesIO(data)).pages).strip()
    if kind == "docx":                                  # a Word document
        docx = need("python-docx", "docx")
        return "\n".join(p.text for p in docx.Document(io.BytesIO(data)).paragraphs).strip()
    raise ValueError(f"Cannot read '{filename}'. Use a .txt, .eml, .msg, .pdf, .docx or .html file.")

# ---- No email yet? Wait for one: pasted or typed text, or a file ----
def ask_email_text():
    """Used by the trigger when the paste block is empty or no file was chosen. Waits for the email."""
    for attempt in range(3):
        print("Paste or type the text of your email in the box below, then press Enter.")
        print("Or press Enter without text to choose a file to upload instead.")
        text = input("Your email: ").strip()
        if text:
            return text
        from google.colab import files               # Colab's upload button
        uploaded = files.upload()
        if uploaded:
            filename, data = next(iter(uploaded.items()))
            return read_email_file(filename, data)
        print("No text and no file. Try again.")
        print()
    raise ValueError("Still no email. Run the trigger block again when you have one.")

# ---- The approved letter as a PDF file ----
def letter_to_pdf(letter, filename):
    """Save the approved letter as a PDF file, and return the file name."""
    import datetime, os
    fpdf = need("fpdf2", "fpdf")
    lines = letter.strip().split("\n")
    if lines and lines[0].startswith("DRAFT"):              # approved: the draft line goes
        lines = lines[1:]
    text = "\n".join(lines).strip()
    pdf = fpdf.FPDF(format="A4")
    pdf.set_margins(25, 25, 25)
    pdf.add_page()
    try:                                                    # a font with all characters (it comes with matplotlib)
        import matplotlib
        folder = os.path.join(matplotlib.get_data_path(), "fonts", "ttf")
        pdf.add_font("DejaVu", "", os.path.join(folder, "DejaVuSans.ttf"))
        pdf.add_font("DejaVu", "B", os.path.join(folder, "DejaVuSans-Bold.ttf"))
        family = "DejaVu"
    except Exception:                                       # otherwise a standard font (simple characters only)
        family = "Helvetica"
        text = text.encode("latin-1", "replace").decode("latin-1")
    pdf.set_font(family, "B", 14)
    pdf.cell(0, 10, "Demo Home Insurance (a fictional insurer)", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font(family, "", 10)
    pdf.cell(0, 6, "Claims desk, " + datetime.date.today().strftime("%d %B %Y"), new_x="LMARGIN", new_y="NEXT")
    pdf.ln(8)
    pdf.set_font(family, "", 11)
    pdf.multi_cell(0, 6, text)
    pdf.ln(10)
    pdf.set_font(family, "", 8)
    pdf.multi_cell(0, 4, "Approved by a claims handler. Teaching example (Applied AI for Business, "
                         "University of Amsterdam): nothing has been sent.")
    pdf.output(filename)
    return filename

# The fixed answers for demo mode. In real mode, the model writes these answers itself.
DEMO_ANSWERS = {
    "claim_1_covered": {
        "intake": "Customer name: Sanne de Vries\nPolicy number: DH-1001\nDate of damage: 3 November 2026\nWhat happened: A hailstorm broke six roof tiles and cracked the skylight above the stairs. Rain is now coming in.\nDamage: Six broken roof tiles and a cracked skylight\nEstimated cost: EUR 1,850",
        "policy": "Decision: COVERED\nReason: Hail damage to the building is covered by policy DH-1001. The deductible of EUR 250 applies.\nPayout: EUR 1,600",
        "letter": "DRAFT - to be checked by a claims handler\n\nDear Sanne de Vries,\n\nThank you for reporting the hail damage of 3 November 2026. We are sorry to hear that rain is coming in.\n\nYour home insurance covers hail damage to the building. Your roofer's quote is EUR 1,850. After your deductible of EUR 250, we will pay EUR 1,600.\n\nYou can ask your roofer to start the repair.\n\nKind regards,\nClaims desk, Demo Home Insurance (a fictional insurer)"
    },
    "claim_2_not_covered": {
        "intake": "Customer name: Mehmet Yilmaz\nPolicy number: DH-2002\nDate of damage: 3 November 2026\nWhat happened: A heavy hailstorm damaged the roof and broke the skylight in the attic.\nDamage: Damaged roof and broken attic skylight\nEstimated cost: EUR 2,400",
        "policy": "Decision: NOT COVERED\nReason: Policy DH-2002 insures only belongings inside the home. Damage to the roof and skylight is damage to the building, which is not insured.\nPayout: EUR 0",
        "letter": "DRAFT - to be checked by a claims handler\n\nDear Mehmet Yilmaz,\n\nThank you for reporting the hail damage of 3 November 2026.\n\nYour policy is a contents insurance. It covers your belongings inside the home, but not the building itself. The roof and the skylight are part of the building, so we cannot pay for this repair.\n\nIf your belongings were damaged as well, please let us know.\n\nKind regards,\nClaims desk, Demo Home Insurance (a fictional insurer)"
    },
    "claim_3_missing_info": {
        "intake": "Customer name: Lisa\nPolicy number: unknown\nDate of damage: unknown\nWhat happened: After a storm last week, water is coming through the ceiling.\nDamage: Water coming through the ceiling\nEstimated cost: unknown",
        "policy": "",
        "letter": ""
    }
}

# A fixed revised letter for demo mode, used when the person says no.
DEMO_REVISED = {
    "claim_1_covered": "DRAFT - to be checked by a claims handler\n\nDear Ms De Vries,\n\nThank you for reporting the hail damage of 3 November 2026. We regret that rain is now coming into your home.\n\nYour home insurance covers hail damage to the building. The roofer's quote is EUR 1,850. After the deductible of EUR 250, we will pay EUR 1,600.\n\nYou may ask your roofer to start the repair. Please send us the final invoice when the work is done.\n\nYours sincerely,\nClaims desk, Demo Home Insurance (a fictional insurer)",
    "claim_2_not_covered": "DRAFT - to be checked by a claims handler\n\nDear Mr Yilmaz,\n\nThank you for reporting the hail damage of 3 November 2026.\n\nYour policy is a contents insurance. It covers your belongings inside the home, but not the building itself. The roof and the skylight are part of the building. We are therefore unable to pay for this repair.\n\nThe owner of the building may have a separate building insurance that covers this damage. If your belongings were also damaged, please let us know.\n\nYours sincerely,\nClaims desk, Demo Home Insurance (a fictional insurer)"
}

print("Engine room ready.")

## Step 6 · Run the team, one step at a time

Now the team goes to work. Each block below is one step. Run them in order.

Each agent block does two things:
1. it writes down a **recipe** for the agent (the lines that start with `def`);
2. it **uses** the recipe once, for the claim you chose (the last lines of the block).

**Watch the boxes.** Every agent prints its answer (its **output**) in a box, with an arrow that says where it goes next. The next agent gets exactly that text as its **input**. Nothing is hidden: what you read is what the next agent reads.

### The trigger: a claim email arrives

**What this block does:** you choose the email in the menu on the right:
- one of the **three example emails**; or
- **my own email: upload**. A *Choose Files* button appears under the block: choose a file (.txt, .eml, .msg, .pdf, .docx or .html). The block waits until you have chosen a file; until then no other block can run. Choose no file, and the block asks you to type or paste the email instead; or
- **my own email: paste**. Paste the text in the block just above the trigger, and run that block. Is that block empty? Then the trigger stops and asks for your email: paste or type it in the box, or press Enter to choose a file.

The block starts a new run and shows the email as the agents will read it.

**Use invented emails only.** In real mode, the email is sent to OpenAI. Never use a real customer's email or your own private mail.

**No email file at hand?** Download a sample email (.eml, .docx, .pdf or .html) from the [sample_emails folder](https://github.com/jorisdemmers/multi-agent-claim-desk/tree/main/data/sample_emails), or write one in Word and save it. From classic Outlook on Windows, drag an email to your desktop: that gives a .msg file. In Gmail, open the email, click *More* (⋮) and then *Download message*: that gives an .eml file.

**Tip:** use one of the two policies in the database: **DH-1001** (Sanne de Vries, home insurance: building and contents) or **DH-2002** (Mehmet Yilmaz, contents only). With another number, agent 2 will not find the policy.

**Demo mode** only has fixed answers for the three example emails. With your own email, you see the email, and then the demo goes on with example 1. Switch to real mode (Step 0) to run your own email.

**Cost:** free.

In [ ]:
# @title Optional: paste your own email here
# Paste your email between the two lines with three quotes. Run this block.
# Then choose "my own email: paste" in the trigger below.
PASTED_EMAIL = """

"""
print("Pasted email:", len(PASTED_EMAIL.strip()), "characters")

In [ ]:
# @title The trigger: choose a claim email
CLAIM = "claim_1_covered"  # @param ["claim_1_covered", "claim_2_not_covered", "claim_3_missing_info", "my own email: upload", "my own email: paste"]

reset_budget()                 # a new run: the budget guard starts counting from zero

if CLAIM == "my own email: upload":
    from google.colab import files               # Colab's upload button
    print("Choose a file: .txt, .eml, .msg, .pdf, .docx or .html")
    uploaded = files.upload()                    # a button appears below; you pick a file on your computer
    if not uploaded:                             # no file chosen: wait for the email as text, or another file
        uploaded = {"my_email.txt": ask_email_text().encode("utf-8")}
    filename, data = next(iter(uploaded.items()))
    CLAIMS["my_email"] = read_email_file(filename, data)   # turn the file into plain text (Step 5)
    CLAIM = "my_email"
elif CLAIM == "my own email: paste":
    if not PASTED_EMAIL.strip():                 # the paste block is empty: wait for the email now
        PASTED_EMAIL = ask_email_text()
    CLAIMS["my_email"] = PASTED_EMAIL
    CLAIM = "my_email"

if len(CLAIMS[CLAIM]) > MAX_EMAIL_CHARS:          # protects the budget: very long emails are cut
    CLAIMS[CLAIM] = CLAIMS[CLAIM][:MAX_EMAIL_CHARS]
    print(f"(A long email: only the first {MAX_EMAIL_CHARS} characters are used.)")

show_box("THE TRIGGER: a claim email arrives (" + CLAIM + ")", CLAIMS[CLAIM],
         "This text goes, unchanged, to agent 1 (intake)")

if DEMO_MODE and CLAIM not in DEMO_ANSWERS:
    print()
    print("Demo mode only has fixed answers for the three example emails.")
    print("To run YOUR email: switch to real mode in Step 0, run Steps 0 to 5 again, then this block.")
    print()
    CLAIM = "claim_1_covered"
    show_box("For now, the demo goes on with this example email (claim_1_covered)", CLAIMS[CLAIM],
             "This text goes, unchanged, to agent 1 (intake)")

### Agent 1 · intake: reads the email and lists the facts

**What happens:** the program sends the model two things: the intake instructions and the email. The model answers with the facts, in six plain lines.

**That answer is passed on unchanged:** the check reads it, and agent 2 gets it as its input.

**What you will see:** agent 1's output in a box, with where it goes next. *unknown* means: the email does not say.

**Cost in real mode:** 1 call to the model.

In [ ]:
# @title Agent 1 · intake

def intake_agent(claim):
    """The recipe for agent 1: send its instructions and the email to the model; return its answer."""
    if DEMO_MODE:
        return DEMO_ANSWERS[claim]["intake"]                            # demo: a fixed example answer
    answer = ask_model("intake agent",                                  # who is asking (for the cost list)
                       INTAKE_INSTRUCTIONS,                             # the agent's instructions
                       [{"role": "user", "content": CLAIMS[claim]}])    # the work: the email
    return answer.output_text                                           # the model's answer, as text

# Use the recipe for the claim you chose:
if CLAIM not in CLAIMS:
    raise ValueError("The trigger did not finish. Run the trigger block again first.")
print("INPUT for agent 1: the email from the trigger.")
facts = intake_agent(CLAIM)
show_box("OUTPUT of agent 1 (intake): the facts", facts,
         "This text goes, unchanged, to the check, and then to agent 2 (policy)")

### The check: are the facts complete?

**What happens:** plain code, no AI, reads agent 1's lines and checks that the required facts are there. For the demo, only the **policy number** is required: without it, the tool cannot find the policy. If it is missing, the chain stops here and a person takes over. Other missing facts are noted, and the chain goes on.

This is the lesson from the lecture: mistakes multiply over the steps, and **a check between the steps breaks the chain**.

**What you will see:** *OK*, or a list of what is missing.

**Cost:** free (no AI).

In [ ]:
# @title The check (plain code, no AI)

# Which facts must be there before agent 2 may start. For the demo, only the policy number:
# without it, the tool cannot find the policy. Want a stricter check? Add labels to the list,
# for example "date of damage" or "estimated cost", and run this block again.
REQUIRED = ["policy number"]

def check_facts(facts):
    """The recipe for the check: read agent 1's lines, and list the required facts that are missing."""
    found = read_lines(facts)                        # "Policy number: DH-1001" becomes a label and a value
    problems = []
    for label in REQUIRED:
        if is_unknown(found.get(label, "")):         # empty or "unknown" counts as missing
            problems.append(label)
    return problems

print("INPUT for the check: the output of agent 1.")
problems = check_facts(facts)
if problems:
    print("STOP. Missing:", ", ".join(problems))
    print("The chain stops here. A person contacts the customer. Agents 2 and 3 will not run.")
else:
    print("OK. The required facts are there. Agent 1's output goes on to agent 2, unchanged.")
    unknown = [label for label, value in read_lines(facts).items() if is_unknown(value)]
    if unknown:
        print("Note: also unknown:", ", ".join(unknown) + ". Agent 2 may ask for more information.")

### Agent 2 · policy: uses the tool, then decides

**What happens, step by step:**
1. The program sends the model the policy instructions, **agent 1's output** (the facts), and the description of the tool.
2. The model answers with a **request**: *please run lookup_policy for policy DH-1001*.
3. Our program runs the tool and sends the result back.
4. The model reads the policy and gives its decision, in three plain lines.

**What you will see:** the tool request, the tool's answer, and agent 2's output in a box. That output goes to agent 3.

**Cost in real mode:** 2 calls to the model (one before the tool, one after).

In [ ]:
# @title Agent 2 · policy (with the tool)

def policy_agent(facts):
    """The recipe for agent 2: give the model agent 1's facts and the tool; return its decision."""
    if DEMO_MODE:
        policy_number = read_lines(facts)["policy number"]
        print("The policy agent asks for the tool: lookup_policy('" + policy_number + "')")
        print("The tool answers:", lookup_policy(policy_number).get("product", "no policy found"))  # the real tool still runs
        return DEMO_ANSWERS[CLAIM]["policy"]                                                      # demo: a fixed decision

    conversation = [{"role": "user", "content": facts}]               # what the agent gets: agent 1's output
    for attempt in range(3):                                          # at most 3 rounds, then give up
        answer = ask_model("policy agent", POLICY_INSTRUCTIONS, conversation, tools=[LOOKUP_POLICY_TOOL])
        conversation += answer.output                                 # remember what the model said
        requests = [item for item in answer.output if item.type == "function_call"]
        if not requests:                                              # no tool request: so this is the decision
            return answer.output_text
        for request in requests:                                      # the model ASKS ...
            policy_number = json.loads(request.arguments)["policy_number"]
            print("The policy agent asks for the tool: lookup_policy('" + policy_number + "')")
            result = lookup_policy(policy_number)                     # ... and OUR CODE runs the tool
            print("The tool answers:", result.get("product", result))
            conversation.append({"type": "function_call_output",      # send the tool's answer back
                                 "call_id": request.call_id,
                                 "output": json.dumps(result)})
    raise RuntimeError("The policy agent did not decide after 3 rounds.")

# Use the recipe, but only if the check said OK:
if problems:
    decision = None
    print("Skipped: the check stopped the chain.")
else:
    print("INPUT for agent 2: the output of agent 1, unchanged.")
    decision = policy_agent(facts)
    show_box("OUTPUT of agent 2 (policy): the decision", decision,
             "This text goes, together with agent 1's facts, to agent 3 (letter)")

### Agent 3 · letter: writes the reply

**What happens:** the glue puts **agent 1's output** (the facts) and **agent 2's output** (the decision) together. That is agent 3's input. The model writes a draft reply.

**What you will see:** agent 3's input (the two outputs together) and its output: the draft letter.

**Cost in real mode:** 1 call to the model.

In [ ]:
# @title Agent 3 · letter

def letter_input(facts, decision):
    """The glue puts the two outputs together. This is exactly what agent 3 gets."""
    return ("FACTS (output of agent 1, the intake agent):\n" + facts.strip() +
            "\n\nDECISION (output of agent 2, the policy agent):\n" + decision.strip())

def letter_agent(facts, decision):
    """The recipe for agent 3: send its instructions and the facts plus the decision; return the draft letter."""
    if DEMO_MODE:
        return DEMO_ANSWERS[CLAIM]["letter"]                            # demo: a fixed example letter
    answer = ask_model("letter agent", LETTER_INSTRUCTIONS,
                       [{"role": "user", "content": letter_input(facts, decision)}])
    return answer.output_text                                           # the letter, as plain text

# Use the recipe, but only if there is a decision:
if decision is None:
    letter = None
    print("Skipped: there is no decision.")
else:
    show_box("INPUT for agent 3 (letter): the outputs of agent 1 and agent 2, put together",
             letter_input(facts, decision))
    letter = letter_agent(facts, decision)
    print()
    show_box("OUTPUT of agent 3 (letter): the draft reply", letter,
             "This text goes to the person (you), who approves it or sends it back")

### The person: you decide

**What happens:** nothing is sent automatically. You are the claims handler. You read the draft and decide.

- Type **yes**: the letter is approved. It is saved as a PDF file, and your browser downloads it.
- Type **no**: you say what should change. The letter goes **back to agent 3** with your feedback, and agent 3 writes a new draft. Then you decide again.
- This can repeat **at most 2 times**. If the letter is still not right, the claims handler writes it themselves.

This is a common pattern: one party judges, the other improves. Here **you** are the judge and the AI improves. It is human in the loop: the AI does the work, a person keeps control.

If you type no, you see exactly what goes back to agent 3: the facts, the decision, its old draft and your feedback.

**What you will see:** the letter and a box to type in. If you type no, a second box asks what should change.

**Cost in real mode:** each new draft is 1 extra call to the model. In demo mode, the new draft is a fixed example and ignores your feedback.

In [ ]:
# @title The person: approve, or send it back with feedback

MAX_REVISIONS = 2   # how often the letter may go back to agent 3

# Extra instructions for agent 3 when its letter comes back with feedback.
REVISION_INSTRUCTIONS = LETTER_INSTRUCTIONS + """

You also get your previous draft and feedback from a claims handler.
Write a new draft that follows the feedback. Do not change the decision or the payout."""

def revision_input(facts, decision, previous_letter, feedback):
    """What agent 3 gets the second time: the same facts and decision, its old draft, and your feedback."""
    return (letter_input(facts, decision) + "\n\nYOUR PREVIOUS DRAFT:\n" + previous_letter.strip() +
            "\n\nFEEDBACK FROM THE CLAIMS HANDLER:\n" + feedback.strip())

def revise_letter(facts, decision, previous_letter, feedback):
    """The recipe for a new draft: send agent 3 its old letter and the person's feedback."""
    if DEMO_MODE:
        print("(Demo mode: a fixed revised letter. In real mode, agent 3 follows your feedback.)")
        return DEMO_REVISED[CLAIM]
    answer = ask_model("letter agent", REVISION_INSTRUCTIONS,
                       [{"role": "user", "content": revision_input(facts, decision, previous_letter, feedback)}])
    return answer.output_text

if letter is None:
    print("Nothing to approve: the chain stopped earlier. A person contacts the customer.")
else:
    revisions = 0                                  # how many new drafts so far
    while True:                                    # "while True" means: repeat until we say stop ("break")
        print(letter)
        print()
        choice = input("Approve this letter? Type yes or no, then press Enter: ").strip().lower()
        if choice == "yes":
            print("Approved. In a real system, the letter would now be sent. Here, nothing is sent.")
            break                                  # stop repeating
        if revisions == MAX_REVISIONS:
            print(f"Still not approved after {MAX_REVISIONS} new drafts. The claims handler writes the letter themselves.")
            break                                  # stop repeating
        feedback = input("What should change? Type your feedback, then press Enter: ").strip()
        print()
        show_box("INPUT for agent 3 (again): facts, decision, its old draft and your feedback",
                 revision_input(facts, decision, letter, feedback))
        letter = revise_letter(facts, decision, letter, feedback)    # agent 3 writes a new draft
        revisions += 1
        print("-" * 60)
    if choice == "yes":                            # approved: the letter becomes a PDF file
        pdf_file = letter_to_pdf(letter, "approved_letter_" + CLAIM + ".pdf")
        print("The approved letter is saved as a PDF:", pdf_file)
        try:
            from google.colab import files
            files.download(pdf_file)               # your browser downloads the PDF
        except ImportError:
            print("(Not in Colab: the file is in the same folder as this notebook.)")

### What did this run cost?

**What you will see:** one line per call to the model, and the total. In demo mode there were no calls, so the cost is zero.

In [ ]:
# @title The cost of this run

print(f"{'Agent':<15}{'Tokens read':>13}{'Tokens written':>16}{'Cost (USD)':>13}")
for agent, read, written, cost in calls:
    print(f"{agent:<15}{read:>13}{written:>16}{cost:>13.5f}")
print(f"Total: {len(calls)} calls, ${total_cost():.5f}")

## Step 7 · The glue: the whole team in one go

In Step 6 you ran the agents one by one, so you could see each step. In a real system, a few lines of code run them in a row. That code is the **glue**.

**What this block does:** it runs one claim from start to finish, using the recipes from Step 6. The person step is left out: the draft waits for approval. Choose *my_email* to run the email you uploaded or pasted in Step 6 (real mode).

**What you will see:** each agent's output in a box, with where it goes next, and the cost.

**Cost in real mode:** up to 4 calls, well under 1 cent.

Notice that the order is fixed in the code: intake, check, policy, letter. The model never decides who goes next. That makes this a **workflow**. If a model decided the next step, it would be an **agent** in the strict sense.

In [ ]:
# @title Step 7 · The glue
RUN_CLAIM = "claim_2_not_covered"  # @param ["claim_1_covered", "claim_2_not_covered", "claim_3_missing_info", "my_email"]

def run_claim_desk(claim):
    """The glue: the trigger, then the agents and the check, in a fixed order."""
    global CLAIM
    CLAIM = claim                                  # the demo answers look up the claim by this name
    reset_budget()
    show_box("TRIGGER: a claim email arrives (" + claim + ")", CLAIMS[claim], "goes to agent 1")
    facts = intake_agent(claim)                    # agent 1
    show_box("AGENT 1 (intake): output", facts, "goes to the check and agent 2")
    problems = check_facts(facts)                  # the check
    if problems:
        print("CHECK     STOP. Missing:", ", ".join(problems), "-> a person takes over.")
        return
    print("CHECK     OK -> agent 1's output goes on to agent 2")
    decision = policy_agent(facts)                 # agent 2 (with the tool)
    show_box("AGENT 2 (policy): output", decision, "goes to agent 3, together with agent 1's output")
    letter = letter_agent(facts, decision)         # agent 3
    show_box("AGENT 3 (letter): output", letter, "goes to the person")
    print("PERSON    the draft waits for a claims handler. Nothing has been sent.")
    print(f"COST      {len(calls)} calls, ${total_cost():.5f}")

if RUN_CLAIM not in CLAIMS:
    print("There is no own email yet. Use the trigger in Step 6 first (my own email: upload or paste).")
elif DEMO_MODE and RUN_CLAIM not in DEMO_ANSWERS:
    print("Demo mode only has fixed answers for the three example emails. Switch to real mode to run your own email.")
else:
    run_claim_desk(RUN_CLAIM)

## Try this

Work in pairs. Write down your prediction **before** you run anything.

1. **Claim 3 stops at the check.** Why? Add the missing facts to the email in Step 2, run Step 2 again, and run the team again (real mode). Does it get through now?
2. **Send the letter back.** At the person step, type *no* and ask for a change, for example *mention the repair deadline* (real mode). Does agent 3 follow your feedback? Does it keep the decision the same?
3. **Change one sentence** in the letter instructions (Step 3), for example: *write the letter in Dutch* or *write in a very formal style*. Run Step 3 and agent 3 again. What changes?
4. **Make the check stricter.** In the check block, add "date of damage" to `REQUIRED` and run the block again: which emails stop now? Next, stop the chain when the estimated cost is above EUR 5,000, so a person must decide on large claims. The check gets the line *Estimated cost: EUR 1,850* from agent 1. (Hint: ask ChatGPT or Claude how to turn that line into a number and add the rule, and paste it in.)
5. **Run the same claim five times** in real mode (Step 7). Is the decision the same every time? Is the letter the same? What does that mean for a firm?
6. **Add a fourth agent:** a fraud agent that runs after the check and flags claims that look odd. You need new instructions (Step 3), a recipe like the other agents (Step 6), and one extra line in the glue (Step 7).
7. **Discuss:** which steps would you never let an agent do without a person? Which level of autonomy (L1 to L5, from the lecture) does this claim desk have?
8. **Your own email** (real mode). Write a claim email yourself, for example in Word, and upload it in the trigger. Try a clear one, a vague one, and one with a policy number that is not in the database. Follow the boxes: where does it go wrong first, and does the check catch it?

## What you have seen

| Lecture term | Where in this notebook |
|---|---|
| Agent = instructions + model + tools | Step 3 (instructions), Step 1 (model), Step 4 (tool) |
| Each agent's output is the next agent's input | The boxes in Step 6 |
| The model asks, the code acts | Agent 2: the tool request and the tool's answer |
| A check breaks the chain of mistakes | The check block |
| Human in the loop | The person block |
| One party judges, the other improves | The person block: type no and give feedback |
| Workflow: the code decides the order | Step 7, the glue |
| Budget and safety | Step 0 (secret), Step 1 (budget), Step 5 (budget guard) |

*Teaching material for Applied AI for Business, Amsterdam Business School, University of Amsterdam. The insurer, its policies and its customers are fictional. This notebook is not connected to any real insurer.*